# Flip reader — reasoned rerun (both setups, reasoning visible)

The 143 run2 flip questions + 50 sampled both-wrong, rerun with `FA_REASONED=1` so
**every turn shows its reasoning**. Buckets below are computed fresh from THIS rerun
(the reasoned prompt changes outcomes vs run2 — that fragility is itself a finding).

How to read: each question shows the question/options/gold, then the two traces
side by side — **left = skim-only**, **right = with tools** (action per round in the
header). To see the actual frames for any question:
`viz.show(T, qid="...")` (tools run) or `viz.show(B, qid="...")` (skim-only run).


In [1]:
import html, os
from IPython.display import HTML, Markdown, display
from fast_agent import config, viz

B = viz.load_run(os.path.join(config.RUN_ROOT, "reasoned_flips_baseline"))
T = viz.load_run(os.path.join(config.RUN_ROOT, "reasoned_flips_tools"))
bby = {t["question_id"]: t for t in B}
tby = {t["question_id"]: t for t in T}
common = [q for q in tby if q in bby]
buckets = {"tools_lost": [], "tools_won": [], "both_wrong": [], "both_right": []}
for q in sorted(common):
    b, t = bby[q]["correct"], tby[q]["correct"]
    key = ("both_right" if b and t else "both_wrong" if not (b or t)
           else "tools_won" if t else "tools_lost")
    buckets[key].append(q)
print({k: len(v) for k, v in buckets.items()}, f"(paired on {len(common)})")

def _rounds_html(traj):
    out = []
    for r in traj["rounds"]:
        a = r.get("action") or {}
        if a.get("kind") == "tool_call":
            lab = f"{a['name']}({a['start']:.0f}-{a['end']:.0f}s)"
            if a.get("error"): lab += " REJECTED"
        else:
            lab = a.get("kind", "?")
        th = html.escape((r.get("thinking") or "").strip())
        out.append(
            f"<div style='margin:4px 0'><b>R{r['round']} [{html.escape(lab)}]</b>"
            f"<pre style='white-space:pre-wrap;font-size:11.5px;margin:2px 0;"
            f"max-height:260px;overflow:auto;background:#00000010;padding:5px'>{th}</pre></div>")
    return "".join(out)

def read(qid):
    b, t = bby[qid], tby[qid]
    mark = lambda x: "✅" if x["correct"] else f"❌({x['pred']})"
    opts = "".join(f"<div>{'➡️ ' if o[:1]==t['gold'] else '&nbsp;&nbsp;&nbsp;'}{html.escape(o)}</div>"
                   for o in t["options"])
    display(HTML(
        f"<div style='border:1px solid #888;border-radius:6px;padding:8px;margin:10px 0;"
        f"font-family:sans-serif'>"
        f"<b>[{qid}] {html.escape(t['task_type'])}</b> · {t['duration']:.0f}s · gold={t['gold']} · "
        f"skim-only {mark(b)} · tools {mark(t)}"
        f"<div style='margin:5px 0'>{html.escape(t['question'])}</div>{opts}"
        f"<table width='100%' style='table-layout:fixed'><tr>"
        f"<td width='50%' valign='top' style='border-right:1px solid #ccc;padding-right:6px'>"
        f"<b>SKIM-ONLY</b>{_rounds_html(b)}</td>"
        f"<td width='50%' valign='top' style='padding-left:6px'>"
        f"<b>WITH TOOLS</b>{_rounds_html(t)}</td>"
        f"</tr></table></div>"))

{'tools_lost': 24, 'tools_won': 29, 'both_wrong': 105, 'both_right': 34} (paired on 192)


## TOOLS LOST IT — skim-only right, tools wrong

In [2]:
print(len(buckets["tools_lost"]), "questions:", ", ".join(buckets["tools_lost"]))
for q in buckets["tools_lost"]:
    read(q)

24 questions: 608-1, 655-1, 655-2, 682-3, 702-1, 719-3, 733-1, 736-1, 758-1, 762-3, 767-2, 786-3, 799-1, 810-2, 811-3, 826-2, 834-3, 838-1, 840-3, 862-3, 875-3, 885-1, 898-2, 900-1


"WITH TOOLSR0 [answer]The question asks which statement about the seven wonders of the ancient world is not correct, based on the video. The video shows various ancient structures, including the Great Pyramid of Giza, the Lighthouse of Alexandria (implied by the underwater ruins and the tower reconstruction), and the statue of Zeus at Olympia (implied by the underwater statue and the reconstruction). It also shows the ruins of the Temple of Artemis and the Mausoleum at Halicarnassus, which are in Turkey, not Egypt. The video focuses heavily on Egypt, showing the Nile, desert ruins, and underwater sites, but it does not show the Hanging Gardens of Babylon or the Colossus of Rhodes. The video does not show the Great Pyramid of Giza as the only surviving wonder, as it shows other structures like the Lighthouse of Alexandria and the statue of Zeus. The video does not show two wonders in Egypt that have not been found yet, as it shows the Great Pyramid of Giza and the Lighthouse of Alexandria, which are both known to exist. The video does show two wonders in Egypt, the Great Pyramid of Giza and the Lighthouse of Alexandria. Therefore, the statement that is not correct is B. Only the Giza pyramid survives. <answer>B</answer>"


"SKIM-ONLYR0 [answer]The video frames show scenes of devastation from tsunamis, including people fleeing, buildings destroyed, and waves crashing. It also includes maps and animations illustrating tsunami propagation. However, none of the frames provide any information about the ""breaking down of the alarm bell"" in Hilo in 1960. The video focuses on the natural causes (terrain, wave impact) and human response, but there is no visual evidence or mention of a malfunctioning alarm system. Therefore, the breaking down of the alarm bell is not depicted as a cause in the video. <answer>C</answer>","WITH TOOLSR0 [compress_video(0-3000s)]The question asks which element was NOT a cause of the great loss in Hilo Island in 1960, based on the video. The video shows scenes of destruction, people fleeing, and interviews with survivors or experts. It also includes maps and animations of tsunamis. To determine the correct answer, I need to identify which of the four options (A, B, C, D) is not supported by the visual evidence. The video likely shows the terrain, the alarm system, and the underestimation of the tsunami's impact, but I need to verify if ""local people's curiosity"" is depicted as a cause. I will use the `compress_video` tool to get a broad overview of the video to locate relevant scenes. <tool_call> {""name"": ""compress_video"", ""arguments"": {""start_time"": 0, ""end_time"": 3000}} </tool_call>R1 [answer]The video is a documentary about tsunamis, showing their destructive power through animations, historical footage, and interviews with survivors and experts. It covers events in Hilo (1960), Japan (1960), Sumatra (2004), and Alaska (1958), highlighting the role of natural terrain, underestimation of impact, and failures in warning systems. The video does not show or imply that ""local people's curiosity"" was a cause of the loss in Hilo, as the focus is on the natural disaster's scale and the failures in preparedness and response. <answer>B</answer>"


"SKIM-ONLYR0 [answer]The video shows a timeline of Galaxy S phones from 2010 to 2024, with the ""Starting Price"" chart indicating that the most expensive phone in this series was the 2022 model, priced at $999. The presenter then shows the 2022 Galaxy S22, which has a white back cover. The question asks for the color of the back cover of the *previous generation* phone, which would be the 2021 Galaxy S21. The video shows the S21 in a blue color, but it does not show the back cover of the S21 in any frame. The video does show the back cover of the S22 (white) and the S23 (blue), but not the S21. Therefore, the information required to answer the question is not present in the provided frames. <answer>B</answer>","WITH TOOLSR0 [answer]The question asks for the color of the back cover of the previous generation Galaxy S phone that had the most expensive starting price shown in the video. The video displays a timeline graph for ""Starting Price"" from 2010 to 2024, and the highest price point shown is $999, which corresponds to the year 2022. The phone associated with this price is the Galaxy S22. The video then shows a comparison between the Galaxy S22 and its predecessor, the Galaxy S21, which is shown with a blue back cover. The question asks for the color of the *previous generation* phone with the *most expensive starting price*, which is the Galaxy S21. The video shows the Galaxy S21 with a blue back cover. Therefore, the correct answer is C. Purple. <answer>C</answer>"


"SKIM-ONLYR0 [answer]The frames show a man with a bandage on his hand holding an envelope, and later, a scene where he is seen with a dog. The dog is shown being held on a leash, and the man is seen walking with it. There is no visual evidence of a firework, a fire, or a motorcycle chase. The frames do not show the man being dragged by a dog or any interaction with Wayne's father. The injury appears to be from a fall or an accident, but the specific cause is not shown in the provided frames. <answer>D</answer>","WITH TOOLSR0 [crop_video(1800-1900s)]The question asks how the man with the bandage and envelope sustained his injury. The video shows him in a scene where he is holding an envelope and has a bandage on his arm. To determine the cause of his injury, I need to find the scene where he is injured and see what happened. The video shows him in a dark setting, possibly at night, and there is a motorcycle nearby. I need to find the specific moment when he got hurt. <tool_call> {""name"": ""crop_video"", ""arguments"": {""start_time"": 1800, ""end_time"": 1900}} </tool_call>R1 [answer]The frames from 1800s to 1900s show a man in a plaid shirt and beanie setting off a firework or similar device, which explodes and sends sparks flying. He is then seen falling to the ground, and later, he is shown with a bandage on his arm while holding an envelope. This sequence directly shows the injury occurring during the firework incident. The other options are not supported by the visual evidence. <answer>A</answer>"


...

"SKIM-ONLYR0 [answer]The frames show the train journey starting in Cusco at 11,152 feet (Day 1) and reaching 14,000 feet (Day 3), explicitly highlighting the high altitude. The scenery includes mountains, valleys, and deserts, but the video also shows the couple enjoying meals, sleeping in their cabin, and even visiting a lake and a town. The train is the primary mode of transport throughout the journey, as shown by the numerous interior shots of the train car and the views from the windows. While the couple is clearly traveling together, there is no indication this is a honeymoon. The most consistent and emphasized aspect across the frames is the train journey through diverse and often high-altitude landscapes. <answer>C</answer>","WITH TOOLSR0 [answer]The video documents a train journey through Peru, starting in Cusco at 11,152 feet and reaching 14,000 feet in Saracocha. Multiple frames show the train traveling through high-altitude mountainous terrain, with text overlays explicitly stating altitudes. The travelers are seen enjoying the views from the train, and the journey is described as reaching the ""highest point"" of the trip. While they do visit other locations like Lake Titicaca and Lima, the core experience is the train ride through varied and elevated landscapes. The video does not focus on uninhabited areas, nor is it explicitly a honeymoon trip. The majority of the video is indeed spent on the train, showcasing the scenery and amenities. The most accurate description of what's special about the trip is that it is a high-altitude journey, as repeatedly emphasized by the altitude markers and the stunning mountain views. <answer>A</answer>"


"SKIM-ONLYR0 [answer]The video shows leopards climbing trees (A), hyenas scavenging or competing for kills (B), and lions hunting or feeding (C). There are no frames showing eagles eating animal carcasses. While vultures are shown feeding on a carcass, eagles are not depicted in any of the provided frames. Therefore, option D is not depicted. <answer>D</answer>","WITH TOOLSR0 [answer]The question asks which option is NOT depicted in the video. Let's analyze each option based on the provided frames. - Option A: ""Leopard climbs on trees."" This is shown in frames 13 and 14, where a leopard is seen climbing a tree and then resting on a rocky outcrop. - Option B: ""Hyenas prey animals."" This is shown in multiple frames, such as 9, 10, 11, 12, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64. Hyenas are consistently shown hunting, running, and feeding on carcasses, which constitutes preying on animals. - Option C: ""A lion is attacked by a group of lions."" This is not depicted. The video shows lions hunting, feeding, and interacting with each other, but there is no scene where a lion is attacked by other lions. Lions are social predators, but they do not typically attack each other in this manner during the events shown. - Option D: ""Eagles eat the bodies of animals."" This is not shown. While the video shows vultures (which are scavengers) feeding on carcasses in frames 16 and 17, there is no depiction of eagles eating animal bodies. Both C and D are not depicted. However, the question asks for the option that is ""not depicted,"" implying a single best answer. Since the video clearly shows vultures feeding, option D is less likely to be the correct answer. Option C, however, is definitively not shown. The video shows lions hunting and feeding, but never a lion being attacked by other lions. Therefore, C is the correct answer. <answer>C</answer>"


"SKIM-ONLYR0 [answer]The frames show a soccer penalty shootout. In frame 12, a player in a blue jersey (attacking side) is shown falling to the ground after a foul, and the ball is then kicked by the defending side. In frame 13, the defending side scores. This sequence indicates that the defending side committed a foul, which led to the attacking side being awarded a penalty, and the defending side then scored from that penalty. This matches option B. <answer>B</answer>"


"SKIM-ONLYR0 [answer]The frames show the author demonstrating various embroidery stitches. I can identify the following techniques by their names displayed on screen: 1. Satin Stitch, 2. Stem Stitch, 3. Lazy Daisy Stitch, 4. French Knot, 5. Bullion Stitch, 6. Backstitch, 7. Couching Stitch, 8. Split Stitch, 9. Feather Stitch. The video shows the author working on different plant motifs, and each stitch is applied to a specific part of the design. The frames clearly show the execution of these 9 distinct stitches. There is no visual evidence of any other stitch being taught. <answer>D</answer>"


"SKIM-ONLYR0 [answer]The frames show the heroine engaging in various activities including gym workouts (e.g., squats, leg press, bench press, and a ""mini workout"" video), eating healthy meals, and taking selfies. While she does appear to be focused on fitness, there is no explicit statement or visual evidence in any of the frames that directly explains her specific reasoning for doing butt training only once a week. The video does not mention a marathon, nor does it show her schedule or provide any information about muscle recovery time or her personal beliefs on workout frequency. Therefore, none of the options can be confirmed from the visual evidence provided. <answer>B</answer>","WITH TOOLSR0 [answer_no_letter]The question asks why the heroine only does butt training once a week, based on the video evidence. The video shows her doing various activities including gym workouts, eating, and resting. While she is shown doing butt training (e.g., in frames 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64), there is no explicit statement or visual evidence in the provided frames that explains her specific frequency choice. The frames show her in different settings and activities, but none directly address the reasoning behind her weekly schedule for butt training. Therefore, I cannot determine the correct answer from the given evidence. <answer>Unknown</answer>R1 [answer_no_letter]<answer>Unknown</answer>R2 [answer_no_letter]<answer>Unknown</answer>R3 [answer_no_letter]<answer>Unknown</answer>R4 [answer_no_letter]<answer>Unknown</answer>R5 [answer_no_letter]<answer>Unknown</answer>R6 [answer]<answer>Unknown</answer>"


## TOOLS WON IT — skim-only wrong, tools right

In [3]:
print(len(buckets["tools_won"]), "questions:", ", ".join(buckets["tools_won"]))
for q in buckets["tools_won"]:
    read(q)

29 questions: 605-2, 606-1, 616-3, 633-2, 647-3, 663-3, 677-2, 678-3, 692-1, 702-2, 704-3, 727-2, 728-2, 741-2, 770-2, 772-1, 773-1, 778-3, 783-1, 787-3, 794-2, 806-2, 818-2, 830-1, 843-2, 850-2, 860-2, 863-2, 896-3


"SKIM-ONLYR0 [answer]The video visually depicts the historical context leading to the split of Austria-Hungary. It shows maps illustrating the empire's ethnic composition (frame 13), the economic disparity between regions (frame 11, showing factories in one area and wheat in another), and the impact of World War I (frame 8, showing Prussia's annexations, and frame 19, showing the skull and flames symbolizing war's devastation). It also shows the rise of nationalism and the weakening of the empire (frame 1, with crossed rifles and a group of people, frame 10, with the boot crushing people, frame 12, with the fist and skull). The video does not show or imply that the communist revolution in Russia was a direct cause of the split. While Russia's revolution had global repercussions, the video's focus is on internal ethnic tensions, economic inequality, and the effects of WWI within the empire itself. The frames do not mention or depict the Russian Revolution as a factor in the Austro-Hungarian split. <answer>C</answer>","WITH TOOLSR0 [answer]The question asks which of the listed options was NOT a cause of the split of the country (presumably Austria-Hungary). The video visually presents several factors: it shows maps of the empire's ethnic composition (frame 13, 14, 20), depicts industrialization and economic disparity (frame 11, 19), shows the impact of World War I (frame 8, 10, 12, 15, 16, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64), and even references the Russian Revolution (frame 17, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64). The video explicitly links the rise of nationalism and the weakening of the empire to WWI (frame 10, 12, 15, 16, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64). It also shows the economic disparity between the German-speaking core and the Hungarian and Slavic regions (frame 11, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64). The video also shows the rise of nationalist movements and the weakening of the empire (frame 10, 12, 15, 16, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64). However, the video does not show or mention the ""tolerance towards its minorities that makes them strong enough"" as a cause of the split. In fact, the video suggests the opposite: the empire's policy of tolerance and equal rights for minorities (frame 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64) was a factor that led to the rise of nationalist movements and ultimately the empire's collapse. Therefore, this option is not a cause, but rather a consequence or a misinterpretation of the situation. <answer>A</answer>"


"SKIM-ONLYR0 [answer]The first minute of the video (0-60 seconds) consists of frames that show a battle scene with a fallen soldier, a portrait of Napoleon, and a historical image of Martin Luther King Jr. These images are presented without any accompanying narration or context that would explain their relationship or the video's purpose. The video then cuts to a man speaking in front of a brick wall, but this does not occur until after the first minute. The frames do not provide any information about Napoleon's battles, achievements, or how he became successful. They simply show a montage of historical figures and events. Therefore, none of the options A, B, C, or D can be confirmed as the subject of the first minute based on the visual evidence alone. <answer>None of the above</answer>","WITH TOOLSR0 [crop_video(0-60s)]The first minute of the video (0s to 60s) contains several frames that establish the context of Napoleon Bonaparte. It opens with a dramatic battle scene, followed by portraits of Napoleon, and then transitions to historical figures like Martin Luther King Jr. and a map of Sardinia. The video then cuts to a speaker (a man in a white shirt) who appears to be introducing the topic. The content is not yet focused on a summary of battles, a critique, or a detailed explanation of how he became successful; it is setting the stage by showing key figures and events related to Napoleon's life and legacy. The speaker's introduction is likely the key to understanding the video's purpose for the first minute. <tool_call> {""name"": ""crop_video"", ""arguments"": {""start_time"": 0, ""end_time"": 60}} </tool_call>R1 [answer]The frames from 0s to 60s show a montage of historical imagery, including battle scenes, portraits of Napoleon Bonaparte, and other historical figures like Martin Luther King Jr. The video also features a speaker (a man in a white shirt) who appears to be introducing the topic. This initial segment is clearly setting the stage for a historical narrative about Napoleon, likely his rise to power and his military campaigns, rather than a summary of battles, a critique, or a detailed explanation of his success. The content is establishing context and introducing the subject. <answer>B</answer>"


"WITH TOOLSR0 [answer]The question asks what kind of traditional technique or scenic spot the youtuber introduced after the tofu making segment. The video frames show a sequence of traditional Kyoto crafts and sights. After the tofu making process (which includes scenes of soybeans, boiling, pressing, and packaging), the video transitions to a scene of a man carefully arranging stones in a garden, which is identified by the on-screen text as ""Tofuku-ji"". This is followed by shots of the temple's architecture and the youtuber discussing the importance of preserving these traditions. This clearly indicates that after tofu making, the video focuses on a Buddhist temple. Therefore, the correct answer is A. A Buddhist temple. <answer>A</answer>"


"SKIM-ONLYR0 [answer]The video presents a clear contrast between two perspectives on Bitcoin. On one side, it features interviews with financial experts (like Prof. Gregory Lablanc and Prof. Philipp Sandner) who discuss Bitcoin's role in the financial system, its volatility, and its potential as an investment asset, often referencing its price movements and integration with traditional finance. On the other side, it shows scenes of people in developing economies using Bitcoin for transactions, a woman speaking about its potential for empowerment, and a scene of a person using a QR code for payment, suggesting its use as a tool for financial inclusion. The video also shows mining operations, which ties into the environmental debate, but the core contrast is between the financial elite's view of it as an investment and the grassroots view of it as a tool for economic empowerment. The video does not focus on technological innovation vs. environmental impact (B) or stability vs. returns (A) as the primary dichotomy, though these are touched upon. The key difference is the perspective: institutional finance vs. grassroots economic empowerment. <answer>D</answer>","WITH TOOLSR0 [answer]The video presents a clear contrast between two perspectives on Bitcoin. On one side, we see interviews with financial experts like Prof. Gregory Lablanc and Prof. Philipp Sandner, who discuss Bitcoin's role in the financial system, its volatility, and its potential as an investment. On the other side, we see scenes of people in developing areas using Bitcoin for transactions, a woman speaking at a government event, and a text overlay stating ""At the start of 2024, the price of Bitcoin climbed to its highest level since April 2022,"" which highlights its appeal as a speculative asset. The video also shows scenes of mining operations and a text overlay of ""-9% DOW JONES,"" suggesting a focus on financial markets and investment. The core difference is not about technological innovation (B) or environmental impact (B), nor about empowering underbanked communities (D) as a primary theme. The video primarily contrasts the view of Bitcoin as a financial asset for investors with its potential as a tool for economic inclusion, but the most prominent contrast is between its role as a speculative investment and its potential to disrupt traditional financial systems. The question asks for the core difference between contrasting viewpoints, and the most significant contrast shown is between the financial/technical perspective (stability, regulation, market performance) and the potential for widespread adoption and empowerment. However, the video does not strongly emphasize the ""empower underbanked"" angle as a primary contrasting viewpoint. The most accurate summary of the core difference is between the financial/technical perspective and the potential for widespread adoption, which is best captured by option C. <answer>C</answer>"


...

"SKIM-ONLYR0 [answer]The frames show a tiered ranking system for Taylor Swift's outfits, with categories like ""GOD TIER,"" ""holy ground,"" ""gorgeous,"" ""nothing new,"" and ""shoelace we said no."" The ""nothing new"" category is visually represented by images of Taylor Swift in outfits that appear to be repeats or variations of previous looks, such as the blue sequined outfit with fringe (seen in multiple categories) and the red/black sequined outfit. The video does not show any outfit in this category that includes a hat, nor does it show any vintage-inspired designs or minimalist styles. The outfits in this category are characterized by their bold and vibrant colors, which is consistent with the other categories. Therefore, none of the options A, B, or D are supported by the visual evidence. Option C is the only one that is consistent with the visual content, as the outfits in the ""nothing new"" category are indeed bold and vibrant, similar to the other categories. <answer>C</answer>","WITH TOOLSR0 [answer]The question asks for the most distinctive feature that sets the ""Nothing new"" category apart from the others. The video shows a tiered system with categories like ""GOD TIER,"" ""holy ground,"" ""gorgeous,"" ""sleeking new,"" and ""nothing new."" The ""nothing new"" category is visually represented by a single outfit in the video: a black outfit with red, vein-like patterns, which is also shown in a still image. This outfit is not a simple or minimalist design, nor is it vintage-inspired. It is also not defined by bold, vibrant colors. The most distinctive feature is that it is the only category that includes an outfit with a hat, as seen in the frames where the ""nothing new"" category is displayed alongside the outfit. This is a unique visual element that differentiates it from the other categories. <answer>A</answer>"


"SKIM-ONLYR0 [answer]The question asks which clothing item was NOT copied in the context of the plagiarism allegations against Zara. The frames show various items: a Zara shirt with a floral print (frame 13), a Zara dress (frame 15), and a Zara jeans (frame 20). The frames also show high heels (frame 11) and mention a lawsuit against Zara for copying designs, including those of Christian Louboutin (frame 11, 12, 13). However, the frames do not show any evidence that Zara copied the high heels. The high heels are shown as a comparison, but the video does not indicate they were copied. The other items (shirt, dress, jeans) are shown as examples of copied designs. Therefore, the high heels are the item that was not copied according to the video's content. <answer>A</answer>","WITH TOOLSR0 [answer]The question asks which clothing item was NOT copied in the context of the plagiarism allegations against Zara. The video includes several frames showing Zara's alleged copying of other brands, such as high heels (frame 11), a floral dress (frame 13), and a floral shirt (frame 10). However, there is no visual evidence in the provided frames showing that Zara copied jeans. While jeans are shown in the video (frame 20), they are presented as part of the manufacturing process, not as an example of copied design. The frames showing the alleged copying focus on footwear and dresses, not denim. Therefore, based on the visual evidence, jeans are the item that was not shown to be copied. <answer>D</answer>"


"SKIM-ONLYR0 [answer]The frames show Bart's activities after arriving in Salt Lake City. Frame 14 shows him at a table with a glass of orange juice, which is clearly breakfast. Frame 15 shows him in a gym, indicating a workout. Frame 16 shows him outdoors with a group, looking at bison. The sequence of these events is breakfast, then gym, then outdoor bison watching. This matches option C. <answer>C</answer>","WITH TOOLSR0 [answer]The question asks for the correct chronological order of Bart's activities on his first day in Salt Lake City. I need to identify the sequence of these specific events: gym workout, outdoor bison watching, and breakfast. Looking at the provided frames: - Frame 107 (00:00:07:02) shows Bart in a gym, sitting on a bench with dumbbells, indicating a gym workout. - Frame 110 (00:00:07:02) shows Bart at a table in a restaurant, eating breakfast with a glass of orange juice, indicating a breakfast scene. - Frame 112 (00:00:07:02) shows Bart and the intern walking outdoors, with a field of bison visible in the background, indicating outdoor bison watching. Based on the visual evidence, the order is: gym workout (frame 107), then breakfast (frame 110), then outdoor bison watching (frame 112). This matches option B. <answer>B</answer>"


"SKIM-ONLYR0 [answer]The frames show Niko yelling into a megaphone while standing by a body of water, with the subtitle ""NIKO'S TOUCHING ME"" and ""WHERE YOU STOP PACKS"" appearing. Later, he is seen yelling ""NIKO'S TOUCHING ME"" while walking away from the camera. The context of the video is a challenge where teams are completing tasks. The subtitle ""I WANT TO TRY AND SCARE THEM"" appears when Niko is holding the megaphone, and ""TO TRY AND LURE SOME KANGAROOS"" appears in a later frame. This indicates Niko is using the megaphone to scare animals, specifically kangaroos, as part of a challenge. The other options are not supported by the visual evidence. <answer>D</answer>"


"SKIM-ONLYR0 [answer]The frames show a person unboxing and displaying a wide variety of scrapbooking and crafting supplies, including washi tape, buttons, stickers, embossing folders, decorative papers, and 3D die-cut images. The video is titled ""SCRAPBOOK HAUL"" and ""Action Bélgica,"" indicating it's a showcase of purchased items. The person is not demonstrating how to use the items (ruling out A), nor is it an introduction to shopping (ruling out B). It's not a showcase of finished handmade works for sale (ruling out D). The content is focused on sharing the new stickers and embellishments received, which aligns with ""Sticker Share"" (C). <answer>C</answer>","WITH TOOLSR0 [answer]The video begins with a title card ""SCRAPBOOK HAUL Action Bélgica,"" indicating it's a showcase of items purchased, likely from a Belgian store or event. The subsequent frames show a person unboxing and displaying a wide variety of scrapbooking supplies, including washi tape, buttons, stickers, embossing folders, decorative papers, and 3D die-cut images. The person handles each item, showing its packaging and contents, which is characteristic of a haul video where the focus is on presenting the products acquired, not on teaching a crafting technique or selling handmade goods. The video ends with a ""Subscribe!"" call to action, which is common for content creators showcasing their purchases. The video is a haul, not a tutorial, and the items are being shown for display, not for sale. The person is not demonstrating how to make something, nor are they introducing shopping for handmade products in a general sense. The items are clearly scrapbooking supplies, not handmade works being sold. <answer>B</answer>"


## BOTH WRONG

In [4]:
print(len(buckets["both_wrong"]), "questions:", ", ".join(buckets["both_wrong"]))
for q in buckets["both_wrong"]:
    read(q)

105 questions: 601-2, 602-3, 610-2, 611-1, 617-3, 622-2, 628-2, 635-3, 641-1, 643-1, 646-2, 650-3, 651-1, 652-1, 653-1, 656-1, 658-2, 661-3, 666-1, 666-3, 668-3, 669-1, 680-1, 681-1, 683-1, 687-1, 691-3, 692-3, 699-3, 702-3, 707-2, 708-2, 709-2, 712-1, 721-3, 723-2, 723-3, 726-2, 727-1, 732-1, 732-3, 735-1, 737-1, 739-3, 742-1, 742-3, 747-2, 748-3, 750-2, 754-3, 755-2, 758-3, 763-3, 765-3, 767-1, 769-3, 773-2, 774-2, 775-1, 776-3, 778-1, 780-1, 785-1, 794-3, 796-1, 798-3, 800-3, 801-1, 804-3, 805-2, 807-1, 811-1, 815-3, 816-2, 820-1, 822-1, 831-2, 832-3, 836-1, 837-1, 841-1, 842-1, 848-3, 849-3, 852-2, 856-1, 859-1, 859-2, 863-1, 864-2, 865-1, 865-2, 867-3, 868-2, 872-3, 878-3, 879-1, 880-3, 882-1, 882-3, 885-2, 896-1, 897-1, 899-1, 900-3


"SKIM-ONLYR0 [answer]The video presents a narrative about the first powered flight, showing historical footage of the Wright brothers and the Whitehead aircraft, and includes interviews with experts. One key frame (image 23) shows Dr. Tom Crutch, identified as a Senior Curator at the Smithsonian Institute, speaking. Another frame (image 22) shows a newspaper clipping with the headline ""FLYING MACHINE FAILED TO FLY,"" which appears to be about Whitehead. The video also features an interview with a man who seems to be a historian or researcher (image 18), and a frame with a newspaper article about Whitehead (image 21). The video's title card (image 3) is ""FIRST FLIGHT: Conquest of the Air,"" suggesting a focus on the Wright brothers' achievement. However, the video does not explicitly state or show any official position from the Smithsonian Institute that the first motorized flight was invented by Whitehead. The Smithsonian Institute is mentioned in the context of Dr. Crutch, who is an expert, but his specific stance on the Whitehead vs. Wright debate is not stated in the provided frames. The video presents both sides of the historical debate, but does not definitively attribute the invention to Whitehead according to the Smithsonian. <answer>C</answer>","WITH TOOLSR0 [answer]The question asks which person or institution believes that the first motorized flight was invented by Whitehead. The video contains several frames with interviewees and historical documents. One key frame (frame 11) shows Dr. Tom Crutch, identified as a Senior Curator at the Smithsonian Institute, speaking. Another frame (frame 20) shows a newspaper clipping with the headline ""FLYING MACHINE FAILED TO FLY,"" which is part of the historical record being discussed. The video's narrative appears to be centered on the controversy surrounding the first powered flight, with the Smithsonian Institute being a key authority on aviation history. The question is whether the Smithsonian Institute, or one of the other listed individuals, believes Whitehead was the inventor. The video does not explicitly state this belief, but the presence of Dr. Crutch, a Smithsonian curator, and the context of the video's discussion of the Wright brothers versus Whitehead, strongly suggest that the Smithsonian Institute is the institution being referenced in this context. The other options (author, Orville Wright, John Brown) are not presented as institutions or authorities on this specific historical claim in the video. <answer>C</answer>"


"SKIM-ONLYR0 [answer]The video frames provided show a historical timeline of Russia from 3800 BC to 1917, focusing on territorial expansion, key rulers, wars, and social changes. The frames include images of maps, rulers, battles, and events like the Crimean War, the Russo-Japanese War, and the Russian Revolution. However, none of the frames mention or show any information about the world's longest railway line, its builders, its length, its completion date, or any involvement of French loans. The video does not contain any visual evidence or text that would allow us to verify or refute any of the statements in options A, B, C, or D regarding the railway line. Therefore, based solely on the evidence presented in the frames, we cannot determine which statement is not correct. <answer>D</answer>","WITH TOOLSR0 [answer]The question asks which statement about the world's longest railway line is not correct, based on the video evidence. The video covers Russian history from 3800 BC to 1917, including the expansion of the Russian Empire and its infrastructure. The frames show the growth of Russia's territory and the development of its railroads, particularly in the late 19th and early 20th centuries. Frame 1891 (1891) shows a map with the length of Siberia as 5,772 miles (9,289 km), which is the length of the Trans-Siberian Railway. Frame 1900 (1900) shows the railway connecting Russia to Vladivostok and Japan, and frame 1905 (1905) shows the railway extending into China. Frame 1916 (1916) shows the railway line as completed. The video does not mention French loans or any involvement of France in the construction of the Trans-Siberian Railway. Therefore, statement D is not correct. <answer>D</answer>"


"SKIM-ONLYR0 [answer_no_letter]The video frames show scenes related to the Norman Conquest of England, including depictions of William the Conqueror, Norman soldiers, and battles. There are also historical illustrations and maps showing Normandy and England. The frames depict a battle scene with soldiers in armor, a map of Normandy, and a scene with a Norman soldier holding a bird. The video also includes interviews with historians discussing the events. However, the video does not provide any explicit information about the sequence of events after the Battle of Stamford Bridge. The frames do not show Harold's coronation, William's marriage, the Norman army landing at Hastings, or Edward's demise. Therefore, based on the information provided in the video, it is not possible to determine which of the events happened after the battle at Stamford Bridge. <answer>None of the above</answer>R1 [answer_no_letter]<answer>None of the above</answer>R2 [answer_no_letter]<answer>None of the above</answer>R3 [answer_no_letter]<answer>None of the above</answer>R4 [answer_no_letter]<answer>None of the above</answer>R5 [answer_no_letter]<answer>None of the above</answer>R6 [answer]<answer>None of the above</answer>","WITH TOOLSR0 [answer]The question asks which event occurred after the Battle of Stamford Bridge. The video contains historical reenactments and commentary that reference key events in the Norman Conquest of England. The Battle of Stamford Bridge took place in 1066, and the Battle of Hastings occurred shortly after, in October 1066. The video includes scenes of Norman soldiers, maps showing Normandy and France, and interviews with historians. One historian, Dr. David Chandler, is shown discussing the events. The video also shows scenes of Harold Godwinson, who was crowned king in January 1066, and William the Conqueror, who landed his army in southern England in September 1066. The sequence of events is crucial: Harold was crowned king after the Battle of Stamford Bridge, which was fought in September 1066. William's army landed in England in September 1066, after Harold's coronation. Edward the Confessor's death occurred in January 1066, before the Battle of Stamford Bridge. The video shows Harold being crowned, which is the event that happened after the battle at Stamford Bridge. The video also shows William's army landing, which happened after Harold's coronation, but before the Battle of Hastings. The video does not show William's marriage, which occurred before the invasion. Therefore, the correct answer is A. Harold's coronation. <answer>A</answer>"


"SKIM-ONLYR0 [answer]The video frames show multiple close-ups of the sculpture ""Apollo and Daphne"" by Gian Lorenzo Bernini. These frames clearly depict Apollo reaching out towards Daphne, with his hand near her shoulder and arm, indicating a physical attempt to grasp her. The frames also show Daphne's hair and body transforming into leaves and branches, with her fingertips visibly taking root into the ground. The sculpture is shown with both Apollo and Daphne having curly hair. The video does not show any frame where Apollo's hand is not touching Daphne's skin or body, nor does it show any frame where Apollo is not physically touching her. Therefore, the statement ""Apollo didn't touch Daphne's skin physically"" is not true according to the visual evidence presented. <answer>A</answer>","WITH TOOLSR0 [answer]The question asks what is NOT true about the artwork ""Apollo and Daphne"" based on the video. The video provides several close-up views of the sculpture, showing the dynamic pose of Apollo reaching for Daphne, Daphne's body transforming into a tree, and the intricate details of their hair and the leaves. The video also shows a painting of the same myth by Francesco Albani, which visually confirms the narrative of Apollo chasing Daphne. The video does not show any frame where Apollo's hand is physically touching Daphne's skin, as she is in the process of transforming. The video does show Apollo's curly hair and Daphne's fingertips taking root. Therefore, the statement that is NOT true is A. <answer>A</answer>"


...

"SKIM-ONLYR0 [answer]The frames show a soccer penalty shootout event. The player in the black short-sleeved shirt and black trousers is seen in several frames, including one where he is preparing to take a penalty kick. The video also shows a scoreboard and a final score graphic. However, none of the frames show any text, logos, or patterns on the back of his shirt or trousers that would indicate historical achievements, winner status, or intricate designs. The video focuses on the gameplay and the event itself, not on the specific details of the players' clothing. Therefore, none of the options A, B, C, or D are supported by the visual evidence. <answer>D</answer>","WITH TOOLSR0 [answer]The question asks why the video focused on the player wearing a black short-sleeved shirt and black trousers. Looking at the provided frames, this player is seen in several shots, including one where he is standing with others and another where he is interacting with the crowd. However, none of the frames show any text, logos, or patterns on the back of his shirt or trousers that would indicate historical achievements, winner status, or exquisite designs. The video does not provide any visual evidence to support options A, C, or D. The player is simply one of the participants in the event, and the video's focus on him appears to be incidental to the overall activity rather than due to any specific feature of his clothing. Therefore, none of the given options are supported by the visual evidence. <answer>B</answer>"


"SKIM-ONLYR0 [answer]The frames show a person browsing through various sticker sets. The first set shown (around 0:10-0:12) features cute, colorful designs like sloths, cacti, and hot air balloons. The second set (around 0:13-0:15) has a more whimsical, glittery theme with phrases like ""You are so cool"" and ""Feel the magic."" The third set, shown at 0:16-0:18, is a clear plastic sheet with black outlines of flowers, leaves, and a butterfly. This set is distinctly different from the first two because it is monochromatic (black and white outlines) and features botanical elements, unlike the colorful, cartoonish, or glittery designs of the other sets. The video does not provide any information about the price of the stickers. <answer>B</answer>","WITH TOOLSR0 [answer]The question asks what sets apart the third set of stickers from the other two. The video shows three distinct sticker sets being presented. The first set (shown at 0:10-0:11) features cute animal stickers like sloths and unicorns. The second set (shown at 0:12-0:13) has various shapes and icons, including hearts and stars. The third set (shown at 0:14-0:15) is a collection of botanical elements, including cacti, flowers, and leaves. This is clearly different from the animal and general icon themes of the first two sets. The video does not provide any information about the price or color of the third set, so options C and D are not supported. Option A is incorrect because the third set is not animal prints; it is botanical. Therefore, the correct answer is B. <answer>B</answer>"


"SKIM-ONLYR0 [answer]The frames show a League of Legends match, specifically a team fight in the mid-lane area around the river. The players are actively engaging in combat, with multiple champions using abilities and attacking each other. The camera frequently zooms in on the fight, and replay segments are shown, indicating that these are key moments of the game. The players' actions are focused on direct combat, not on retreating or protecting a specific teammate. The frames do not show any clear strategy of safeguarding the ADC or dispersing the opposing forces. The team appears to be engaging in proactive skirmishes, as they are initiating fights and fighting in the mid-lane. <answer>C</answer>","WITH TOOLSR0 [answer]The video shows a League of Legends match, with the team DK (identified by the ""DK"" logo on the players' jerseys and the ""DK"" team name in the game UI) playing against T1. The frames depict various in-game actions, including team fights, skirmishes, and objective fights. The players are shown actively engaging in combat, often with multiple players from DK participating in fights, suggesting a proactive and aggressive playstyle. The team frequently engages in fights around key areas like the river and mid-lane, which is characteristic of a team that seeks to control the map and apply pressure. The presence of ""REPLAY"" indicators suggests that the video highlights key moments of engagement, further emphasizing the team's active involvement in battles. The question asks for the tactics of team DK, and the visual evidence points towards a team that is not passive but rather seeks to initiate and participate in fights. <answer>C</answer>"


"SKIM-ONLYR0 [answer]The frames show Miranda speaking about Sicily, mentioning its architecture, the sea, and its cultural aspects like the ""bestiario messicano"" (Mexican bestiary). She also talks about the ""paesino in cui vivo"" (the little town where I live) and how it's ""spettacolare"" (spectacular). However, none of the frames provide any information about her parents' hometowns or her birthplace. The video does not contain any explicit statement linking her personal background to Sicily. Therefore, while she clearly has a strong connection to Sicily, the provided frames do not support any of the specific options A, B, C, or D. <answer>D</answer>"


"SKIM-ONLYR0 [answer]The frames show a variety of activities: the woman measuring her body, eating meals (oatmeal, breakfast with avocado, snacks), working out in a gym (squats, bench press, leg press), taking selfies, sleeping, and interacting with her boyfriend (eating together, standing in the kitchen). While fitness and eating are prominent, the video also includes scenes of her home environment, her boyfriend, and her daily routine. The video does not focus exclusively on fitness or eating, nor is it primarily about her home life with her boyfriend. It is a compilation of her daily activities, which includes all the elements mentioned in the options. Therefore, none of the options A, B, or C fully capture the primary focus, which is a general daily vlog. <answer>D</answer>","WITH TOOLSR0 [answer]The video contains a mix of scenes: the woman measuring her body, eating meals (oatmeal, breakfast, snacks), working out in the gym, taking selfies, sleeping, and interacting with her boyfriend (e.g., eating together, standing in the kitchen). While fitness and eating are prominent, the video also shows her home environment, interactions with her boyfriend, and daily routines. The most comprehensive description of the content is that it shows her home life and her relationship with her boyfriend, as these elements appear across multiple scenes and are central to the narrative. The fitness and eating aspects are part of her daily routine, but not the sole focus. <answer>C</answer>"


## Frames

For any question above: `viz.show(T, qid="...")` renders the skim + every tool round's returned frames inline.

In [5]:
# example — uncomment and set a qid:
# viz.show(T, qid=buckets["tools_lost"][0])